In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Importing dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
# Handling categorical data
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
# Dropping the 'USER ID' Columns as it is unique identifier
dataset = dataset.drop("User ID",axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [5]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [6]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [7]:
# Feature and target selection
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
# Splitting data into training and test sets
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [11]:
x_train.shape

(280, 3)

In [12]:
x_test.shape

(120, 3)

In [13]:
y_train.shape

(280,)

In [14]:
y_test.shape

(120,)

In [15]:
# Data Standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [16]:
# GridSearch CV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Defining the different parameters for gridsearch optimization
param_grid = {'criterion':['gini','entropy'],
              'n_estimators':[10,50,100],
              'class_weight':['balanced','balanced_subsample']}


grid = GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,scoring='f1_weighted',n_jobs=-1)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced', 'balanced_subsample'],
                         'criterion': ['gini', 'entropy'],
                         'n_estimators': [10, 50, 100]},
             scoring='f1_weighted', verbose=3)

In [17]:
# Best gridsearch parameter
print(grid.best_params_)

{'class_weight': 'balanced_subsample', 'criterion': 'gini', 'n_estimators': 100}


In [18]:
# Grid prediction
y_pred = grid.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1])

In [19]:
# evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[73  6]
 [ 5 36]]


In [20]:
from sklearn.metrics import classification_report
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.94      0.92      0.93        79
           1       0.86      0.88      0.87        41

    accuracy                           0.91       120
   macro avg       0.90      0.90      0.90       120
weighted avg       0.91      0.91      0.91       120



In [21]:
from sklearn.metrics import f1_score
f1_weighted= f1_score(y_test,y_pred,average='weighted')
print(f1_weighted)

0.9085936101092268


In [22]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.952763198518061)

In [23]:
results = grid.cv_results_
table = pd.DataFrame.from_dict(results)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.027011,0.005578,0.006001,0.001103,balanced,gini,10,"{'class_weight': 'balanced', 'criterion': 'gin...",0.874254,0.821429,0.841398,0.892857,0.870721,0.860132,0.025422,10
1,0.100581,0.013987,0.008722,0.000805,balanced,gini,50,"{'class_weight': 'balanced', 'criterion': 'gin...",0.874254,0.875644,0.823129,0.911105,0.926743,0.882175,0.035819,7
2,0.185249,0.010780,0.010753,0.001743,balanced,gini,100,"{'class_weight': 'balanced', 'criterion': 'gin...",0.874254,0.875644,0.841398,0.892857,0.945469,0.885924,0.034112,5
3,0.020474,0.002447,0.004995,0.001538,balanced,entropy,10,"{'class_weight': 'balanced', 'criterion': 'ent...",0.874254,0.782971,0.787755,0.946663,0.945469,0.867422,0.071967,9
4,0.088266,0.003566,0.006949,0.000289,balanced,entropy,50,"{'class_weight': 'balanced', 'criterion': 'ent...",0.892857,0.840114,0.841398,0.911105,0.927778,0.882651,0.035948,6
5,0.201524,0.017891,0.013594,0.004769,balanced,entropy,100,"{'class_weight': 'balanced', 'criterion': 'ent...",0.874254,0.875644,0.841398,0.911105,0.963889,0.893258,0.041637,2
6,0.031801,0.009092,0.004851,0.001641,balanced_subsample,gini,10,"{'class_weight': 'balanced_subsample', 'criter...",0.835985,0.821429,0.824293,0.874356,0.872761,0.845765,0.023217,12
7,0.133055,0.020406,0.010228,0.004254,balanced_subsample,gini,50,"{'class_weight': 'balanced_subsample', 'criter...",0.874254,0.857143,0.841398,0.892857,0.964286,0.885988,0.042745,4
8,0.261942,0.021444,0.012349,0.002508,balanced_subsample,gini,100,"{'class_weight': 'balanced_subsample', 'criter...",0.874254,0.857143,0.859435,0.911105,0.982051,0.896797,0.046796,1
9,0.029815,0.002092,0.005656,0.001482,balanced_subsample,entropy,10,"{'class_weight': 'balanced_subsample', 'criter...",0.835985,0.802399,0.841398,0.892857,0.926743,0.859876,0.044215,11


In [24]:
age = int(input("Enter your age:"))
salary = float(input("Enter your salary"))
gender = int(input("male=1,female=0"))

In [25]:
data = sc.transform([[age,salary,gender]])

C:\Users\thiru\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
result = grid.predict(data)
result

array([1])